# Notebook 01 — Read and Join Olist Tables

This notebook reads the Olist relational tables from PostgreSQL, inspects their structure and relationships, aggregates one-to-many tables, and creates a single machine-learning table.

Goals:
- Read and inspect every Olist table
- Check row counts, columns, keys, and duplicates
- Understand what one row represents in each table
- Validate relationships between tables
- Aggregate one-to-many tables before joining
- Build one ML table with exactly one row per order
- Save the joined table as an artifact for Notebook 02

Artifact:
`artifacts/01_joined/ml_table.parquet`

In [1]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)


def find_project_root():

    current_path = Path.cwd().resolve()

    candidate_roots = [
        current_path,
        *current_path.parents,
    ]

    for candidate in candidate_roots:

        if (
            (candidate / "compose.yaml").exists()
            and
            (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Olist-MLOps project root. "
        "Run Jupyter from inside the project directory."
    )


PROJECT_ROOT = find_project_root()


ARTIFACT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "01_joined"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Project root      :",
    PROJECT_ROOT
)

print(
    "Artifact directory:",
    ARTIFACT_DIR
)

Project root      : G:\(01)04\Qafza_MLOps\Olist-MLOps
Artifact directory: G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\01_joined


In [2]:
DATABASE_URL = (
    "postgresql+psycopg2://"
    "olist_user:olist_password@127.0.0.1:5433/olist"
)

engine = create_engine(
    DATABASE_URL
)

with engine.connect() as connection:

    database_name = connection.execute(
        text(
            "SELECT current_database();"
        )
    ).scalar()

    current_user = connection.execute(
        text(
            "SELECT current_user;"
        )
    ).scalar()

print(
    "Connected to database:",
    database_name
)

print(
    "Connected as user:",
    current_user
)

Connected to database: olist
Connected as user: olist_user


In [3]:
TABLES = [
    "orders",
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "products",
    "sellers",
    "geolocation",
    "product_category_translation",
]

print(
    "Expected tables:",
    len(TABLES)
)

print(TABLES)

Expected tables: 9
['orders', 'customers', 'order_items', 'order_payments', 'order_reviews', 'products', 'sellers', 'geolocation', 'product_category_translation']


In [4]:
database_tables = pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
    """,
    engine,
)

available_tables = set(
    database_tables["table_name"]
)

missing_tables = (
    set(TABLES)
    - available_tables
)

print(
    "Available expected tables:",
    len(set(TABLES) & available_tables)
)

print(
    "Missing expected tables:",
    missing_tables
)

assert not missing_tables, (
    f"Missing tables: {missing_tables}"
)

print(
    "Database table validation passed."
)

Available expected tables: 9
Missing expected tables: set()
Database table validation passed.


In [5]:
tables = {}

for table_name in TABLES:

    print(
        f"Reading: {table_name}"
    )

    tables[table_name] = pd.read_sql(
        f"SELECT * FROM {table_name};",
        engine,
    )

print(
    "\nAll tables loaded successfully."
)

Reading: orders
Reading: customers
Reading: order_items
Reading: order_payments
Reading: order_reviews
Reading: products
Reading: sellers
Reading: geolocation
Reading: product_category_translation

All tables loaded successfully.


In [6]:
table_overview_rows = []

for table_name, df in tables.items():

    table_overview_rows.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "full_row_duplicates":
            int(df.duplicated().sum()),
    })

table_overview = pd.DataFrame(
    table_overview_rows
)

display(table_overview)

,table,rows,columns,full_row_duplicates
0,orders,99441,8,0
1,customers,99441,5,0
2,order_items,112650,7,0
3,order_payments,103886,5,0
4,order_reviews,99224,7,0
5,products,32951,9,0
6,sellers,3095,4,0
7,geolocation,1000163,5,261831
8,product_category_translation,71,2,0


In [7]:
for table_name, df in tables.items():

    print(
        f"\n{'=' * 90}"
    )

    print(
        table_name.upper()
    )

    print(
        "=" * 90
    )

    print(
        f"Shape: {df.shape}"
    )

    print(
        "\nColumns and dtypes:"
    )

    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype":
                df.dtypes.astype(str).values,
        })
    )

    print(
        "\nSample rows:"
    )

    display(
        df.head(3)
    )


ORDERS
Shape: (99441, 8)

Columns and dtypes:


,column,dtype
0,order_id,str
1,customer_id,str
2,order_status,str
3,order_purchase_timestamp,str
4,order_approved_at,str
5,order_delivered_carrier_date,str
6,order_delivered_customer_date,str
7,order_estimated_delivery_date,str



Sample rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



CUSTOMERS
Shape: (99441, 5)

Columns and dtypes:


...

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



PRODUCT_CATEGORY_TRANSLATION
Shape: (71, 2)

Columns and dtypes:


,column,dtype
0,product_category_name,str
1,product_category_name_english,str



Sample rows:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## Table Grain — What One Row Represents

| Table | One row represents |
|---|---|
| `orders` | One order |
| `customers` | One customer record associated with an order |
| `order_items` | One item inside an order |
| `order_payments` | One payment record for an order |
| `order_reviews` | One review record associated with an order |
| `products` | One product |
| `sellers` | One seller |
| `geolocation` | One geographic coordinate record for a ZIP-code prefix |
| `product_category_translation` | One Portuguese-to-English product-category mapping |

The final machine-learning table must have one row per `order_id`.

Tables such as `order_items`, `order_payments`, `order_reviews`, and `geolocation` can contain multiple rows for the same joining key, so they must not be joined directly without checking their grain first.

In [8]:
CANDIDATE_KEYS = {
    "orders": [
        ["order_id"],
    ],

    "customers": [
        ["customer_id"],
    ],

    "order_items": [
        ["order_id", "order_item_id"],
    ],

    "order_payments": [
        ["order_id", "payment_sequential"],
    ],

    "order_reviews": [
        ["review_id", "order_id"],
    ],

    "products": [
        ["product_id"],
    ],

    "sellers": [
        ["seller_id"],
    ],

    "product_category_translation": [
        ["product_category_name"],
    ],
}


key_check_rows = []

for table_name, candidate_keys in CANDIDATE_KEYS.items():

    df = tables[table_name]

    for key_columns in candidate_keys:

        duplicate_count = (
            df.duplicated(
                subset=key_columns
            )
            .sum()
        )

        missing_count = (
            df[key_columns]
            .isna()
            .any(axis=1)
            .sum()
        )

        key_check_rows.append({
            "table": table_name,
            "candidate_key":
                " + ".join(key_columns),
            "rows": len(df),
            "unique_keys":
                len(df.drop_duplicates(
                    subset=key_columns
                )),
            "duplicate_keys":
                int(duplicate_count),
            "missing_key_rows":
                int(missing_count),
        })

key_summary = pd.DataFrame(
    key_check_rows
)

display(key_summary)

,table,candidate_key,rows,unique_keys,duplicate_keys,missing_key_rows
0,orders,order_id,99441,99441,0,0
1,customers,customer_id,99441,99441,0,0
2,order_items,order_id + order_item_id,112650,112650,0,0
3,order_payments,order_id + payment_sequential,103886,103886,0,0
4,order_reviews,review_id + order_id,99224,99224,0,0
5,products,product_id,32951,32951,0,0
6,sellers,seller_id,3095,3095,0,0
7,product_category_translation,product_category_name,71,71,0,0


In [9]:
ORDER_LEVEL_TABLES = [
    "order_items",
    "order_payments",
    "order_reviews",
]

multiplicity_rows = []

for table_name in ORDER_LEVEL_TABLES:

    counts = (
        tables[table_name]
        .groupby("order_id")
        .size()
    )

    multiplicity_rows.append({
        "table": table_name,

        "orders":
            len(counts),

        "mean_rows_per_order":
            counts.mean(),

        "orders_with_multiple_rows":
            int((counts > 1).sum()),

        "max_rows_per_order":
            int(counts.max()),
    })

multiplicity_summary = pd.DataFrame(
    multiplicity_rows
)

display(
    multiplicity_summary
)

,table,orders,mean_rows_per_order,orders_with_multiple_rows,max_rows_per_order
0,order_items,98666,1.141731,9803,21
1,order_payments,99440,1.044710,2961,29
2,order_reviews,98673,1.005584,547,3


In [10]:
RELATIONSHIPS = [
    (
        "orders",
        "customer_id",
        "customers",
        "customer_id",
    ),

    (
        "order_items",
        "order_id",
        "orders",
        "order_id",
    ),

    (
        "order_items",
        "product_id",
        "products",
        "product_id",
    ),

    (
        "order_items",
        "seller_id",
        "sellers",
        "seller_id",
    ),

    (
        "order_payments",
        "order_id",
        "orders",
        "order_id",
    ),

    (
        "order_reviews",
        "order_id",
        "orders",
        "order_id",
    ),
]


relationship_rows = []

for (
    child_table,
    child_key,
    parent_table,
    parent_key,
) in RELATIONSHIPS:

    child_values = (
        tables[child_table][child_key]
        .dropna()
    )

    parent_values = set(
        tables[parent_table][parent_key]
        .dropna()
    )

    missing_references = (
        ~child_values.isin(
            parent_values
        )
    ).sum()

    relationship_rows.append({
        "relationship":
            f"{child_table}.{child_key} -> "
            f"{parent_table}.{parent_key}",

        "missing_references":
            int(missing_references),
    })

relationship_summary = pd.DataFrame(
    relationship_rows
)

display(
    relationship_summary
)

,relationship,missing_references
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0
3,order_items.seller_id -> sellers.seller_id,0
4,order_payments.order_id -> orders.order_id,0
5,order_reviews.order_id -> orders.order_id,0


## Aggregation Plan

`orders` is the base table because the final ML table must contain one row per order.

Before joining:

- `customers`: join at `customer_id`; aggregate geolocation by ZIP prefix first.
- `order_items`: enrich with product, category, seller, and seller-geolocation information, then aggregate by `order_id`.
- `order_payments`: aggregate by `order_id`.
- `order_reviews`: aggregate by `order_id`.
- `geolocation`: aggregate by ZIP-code prefix before joining because one ZIP prefix can have many coordinate rows.

All final joins use `LEFT JOIN` / left merges from `orders` so orders are not silently removed.

In [11]:
orders = pd.read_sql(
    """
    SELECT *
    FROM orders;
    """,
    engine,
)

print(
    "Orders shape:",
    orders.shape
)

print(
    "Unique order_id:",
    orders["order_id"].nunique()
)

assert (
    len(orders)
    == orders["order_id"].nunique()
)

print(
    "Orders grain validation passed."
)

Orders shape: (99441, 8)
Unique order_id: 99441
Orders grain validation passed.


In [12]:
customers_enriched = pd.read_sql(
    """
    WITH geo_zip AS (
        SELECT
            geolocation_zip_code_prefix,
            AVG(geolocation_lat)
                AS customer_lat,
            AVG(geolocation_lng)
                AS customer_lng

        FROM geolocation

        GROUP BY
            geolocation_zip_code_prefix
    )

    SELECT
        c.customer_id,
        c.customer_unique_id,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        g.customer_lat,
        g.customer_lng

    FROM customers c

    LEFT JOIN geo_zip g
        ON c.customer_zip_code_prefix
        = g.geolocation_zip_code_prefix;
    """,
    engine,
)

print(
    "Customers enriched:",
    customers_enriched.shape
)

print(
    "Unique customer_id:",
    customers_enriched[
        "customer_id"
    ].nunique()
)

assert (
    len(customers_enriched)
    == customers_enriched[
        "customer_id"
    ].nunique()
)

Customers enriched: (99441, 7)
Unique customer_id: 99441


In [13]:
items_agg = pd.read_sql(
    """
    WITH geo_zip AS (
        SELECT
            geolocation_zip_code_prefix,
            AVG(geolocation_lat) AS lat,
            AVG(geolocation_lng) AS lng

        FROM geolocation

        GROUP BY
            geolocation_zip_code_prefix
    ),

    item_details AS (
        SELECT
            oi.order_id,
            oi.order_item_id,
            oi.product_id,
            oi.seller_id,
            oi.shipping_limit_date,
            oi.price,
            oi.freight_value,

            p.product_category_name,
            t.product_category_name_english,

            p.product_photos_qty,
            p.product_weight_g,
            p.product_length_cm,
            p.product_height_cm,
            p.product_width_cm,

            s.seller_state,

            sg.lat AS seller_lat,
            sg.lng AS seller_lng

        FROM order_items oi

        LEFT JOIN products p
            ON oi.product_id
            = p.product_id

        LEFT JOIN product_category_translation t
            ON p.product_category_name
            = t.product_category_name

        LEFT JOIN sellers s
            ON oi.seller_id
            = s.seller_id

        LEFT JOIN geo_zip sg
            ON s.seller_zip_code_prefix
            = sg.geolocation_zip_code_prefix
    )

    SELECT
        order_id,

        COUNT(*)
            AS item_count,

        COUNT(DISTINCT product_id)
            AS unique_products,

        COUNT(DISTINCT seller_id)
            AS unique_sellers,

        SUM(price)
            AS total_item_price,

        AVG(price)
            AS avg_item_price,

        SUM(freight_value)
            AS total_freight_value,

        AVG(freight_value)
            AS avg_freight_value,

        MAX(shipping_limit_date)
            AS max_shipping_limit_date,

        COUNT(
            DISTINCT product_category_name
        )
            AS unique_product_categories,

        MODE() WITHIN GROUP (
            ORDER BY
                product_category_name_english
        )
            AS primary_product_category,

        AVG(product_weight_g)
            AS avg_product_weight_g,

        MAX(product_weight_g)
            AS max_product_weight_g,

        AVG(product_length_cm)
            AS avg_product_length_cm,

        AVG(product_height_cm)
            AS avg_product_height_cm,

        AVG(product_width_cm)
            AS avg_product_width_cm,

        AVG(product_photos_qty)
            AS avg_product_photos_qty,

        COUNT(
            DISTINCT seller_state
        )
            AS unique_seller_states,

        MODE() WITHIN GROUP (
            ORDER BY seller_state
        )
            AS primary_seller_state,

        AVG(seller_lat)
            AS avg_seller_lat,

        AVG(seller_lng)
            AS avg_seller_lng

    FROM item_details

    GROUP BY
        order_id;
    """,
    engine,
)

print(
    "Items aggregated:",
    items_agg.shape
)

print(
    "Unique order_id:",
    items_agg[
        "order_id"
    ].nunique()
)

assert (
    len(items_agg)
    == items_agg["order_id"].nunique()
)

Items aggregated: (98666, 21)
Unique order_id: 98666


In [14]:
payments_agg = pd.read_sql(
    """
    SELECT
        order_id,

        COUNT(*)
            AS payment_records,

        COUNT(
            DISTINCT payment_type
        )
            AS payment_types_count,

        SUM(payment_value)
            AS payment_total,

        MAX(payment_installments)
            AS payment_installments_max,

        MODE() WITHIN GROUP (
            ORDER BY payment_type
        )
            AS primary_payment_type

    FROM order_payments

    GROUP BY
        order_id;
    """,
    engine,
)

print(
    "Payments aggregated:",
    payments_agg.shape
)

print(
    "Unique order_id:",
    payments_agg[
        "order_id"
    ].nunique()
)

assert (
    len(payments_agg)
    == payments_agg["order_id"].nunique()
)

Payments aggregated: (99440, 6)
Unique order_id: 99440


In [15]:
reviews_agg = pd.read_sql(
    """
    SELECT
        order_id,

        COUNT(*)
            AS review_count,

        AVG(review_score)
            AS review_score_mean,

        MIN(review_score)
            AS review_score_min,

        MAX(review_score)
            AS review_score_max,

        MAX(
            CASE
                WHEN review_comment_message
                     IS NOT NULL
                THEN 1
                ELSE 0
            END
        )
            AS has_review_comment

    FROM order_reviews

    GROUP BY
        order_id;
    """,
    engine,
)

print(
    "Reviews aggregated:",
    reviews_agg.shape
)

print(
    "Unique order_id:",
    reviews_agg[
        "order_id"
    ].nunique()
)

assert (
    len(reviews_agg)
    == reviews_agg["order_id"].nunique()
)

Reviews aggregated: (98673, 6)
Unique order_id: 98673


### Leakage Note

Review information and actual delivery information are retained in the joined artifact because later notebooks may need them for analysis or label construction.

They must not automatically become predictive model features if they are unavailable at the chosen prediction point.

In [16]:
aggregation_summary = pd.DataFrame({
    "dataset": [
        "orders",
        "customers_enriched",
        "items_agg",
        "payments_agg",
        "reviews_agg",
    ],

    "rows": [
        len(orders),
        len(customers_enriched),
        len(items_agg),
        len(payments_agg),
        len(reviews_agg),
    ],

    "unique_order_or_customer_key": [
        orders["order_id"].nunique(),
        customers_enriched[
            "customer_id"
        ].nunique(),
        items_agg["order_id"].nunique(),
        payments_agg[
            "order_id"
        ].nunique(),
        reviews_agg[
            "order_id"
        ].nunique(),
    ],
})

display(
    aggregation_summary
)

,dataset,rows,unique_order_or_customer_key
0,orders,99441,99441
1,customers_enriched,99441,99441
2,items_agg,98666,98666
3,payments_agg,99440,99440
4,reviews_agg,98673,98673


In [17]:
ml_table = (
    orders

    .merge(
        customers_enriched,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )

    .merge(
        items_agg,
        on="order_id",
        how="left",
        validate="one_to_one",
    )

    .merge(
        payments_agg,
        on="order_id",
        how="left",
        validate="one_to_one",
    )

    .merge(
        reviews_agg,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

In [18]:
orders_rows = len(orders)
ml_rows = len(ml_table)

unique_orders = (
    ml_table["order_id"].nunique()
)

duplicate_orders = (
    ml_table["order_id"]
    .duplicated()
    .sum()
)

print(
    "Orders rows      :",
    orders_rows
)

print(
    "ML table rows    :",
    ml_rows
)

print(
    "Unique order_id  :",
    unique_orders
)

print(
    "Duplicate order_id:",
    duplicate_orders
)

assert (
    ml_rows == orders_rows
), "Join changed the number of orders."

assert (
    unique_orders == ml_rows
), "ML table is not one row per order."

assert (
    duplicate_orders == 0
), "Duplicate order_id detected."

print(
    "\nOne-row-per-order validation passed."
)

Orders rows      : 99441
ML table rows    : 99441
Unique order_id  : 99441
Duplicate order_id: 0

One-row-per-order validation passed.


In [19]:
coverage = pd.DataFrame({
    "column": [
        "item_count",
        "payment_total",
        "review_count",
        "customer_lat",
        "avg_seller_lat",
    ],

    "missing": [
        ml_table[
            "item_count"
        ].isna().sum(),

        ml_table[
            "payment_total"
        ].isna().sum(),

        ml_table[
            "review_count"
        ].isna().sum(),

        ml_table[
            "customer_lat"
        ].isna().sum(),

        ml_table[
            "avg_seller_lat"
        ].isna().sum(),
    ],
})

coverage["missing_pct"] = (
    coverage["missing"]
    / len(ml_table)
    * 100
)

display(
    coverage
)

,column,missing,missing_pct
0,item_count,775,0.779357
1,payment_total,1,0.001006
2,review_count,768,0.772317
3,customer_lat,278,0.279563
4,avg_seller_lat,991,0.996571


In [20]:
print(
    "Final ML table shape:",
    ml_table.shape
)

display(
    ml_table.head()
)

Final ML table shape: (99441, 44)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,item_count,unique_products,unique_sellers,total_item_price,avg_item_price,total_freight_value,avg_freight_value,max_shipping_limit_date,unique_product_categories,primary_product_category,avg_product_weight_g,max_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,avg_product_photos_qty,unique_seller_states,primary_seller_state,avg_seller_lat,avg_seller_lng,payment_records,payment_types_count,payment_total,payment_installments_max,primary_payment_type,review_count,review_score_mean,review_score_min,review_score_max,has_review_comment
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,-23.576983,-46.587161,1.0,1.0,1.0,29.99,29.99,8.72,8.72,2017-10-06 11:07:15,1.0,housewares,500.0,500.0,19.0,8.0,13.0,4.0,1.0,SP,-23.680729,-46.444238,3.0,2.0,38.71,1.0,voucher,1.0,4.0,4.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,-12.177924,-44.660711,1.0,1.0,1.0,118.70,118.70,22.76,22.76,2018-07-30 03:24:27,1.0,perfumery,400.0,400.0,19.0,13.0,19.0,1.0,1.0,SP,-19.807681,-43.980427,1.0,1.0,141.46,1.0,boleto,1.0,4.0,4.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,-16.745150,-48.514783,1.0,1.0,1.0,159.90,159.90,19.22,19.22,2018-08-13 08:55:23,1.0,auto,420.0,420.0,24.0,19.0,21.0,1.0,1.0,SP,-21.363502,-48.229601,1.0,1.0,179.12,3.0,credit_card,1.0,5.0,5.0,5.0,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,-5.774190,-35.271143,1.0,1.0,1.0,45.00,45.00,27.20,27.20,2017-11-23 19:45:59,1.0,pet_shop,450.0,450.0,30.0,10.0,20.0,3.0,1.0,MG,-19.837682,-43.924053,1.0,1.0,72.20,1.0,credit_card,1.0,5.0,5.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,-23.676370,-46.514627,1.0,1.0,1.0,19.90,19.90,8.72,8.72,2018-02-19 20:31:37,1.0,stationery,250.0,250.0,51.0,15.0,15.0,4.0,1.0,SP,-23.543395,-46.262086,1.0,1.0,28.62,1.0,credit_card,1.0,5.0,5.0,5.0,0.0


In [21]:
ARTIFACT_PATH = (
    ARTIFACT_DIR
    / "ml_table.parquet"
)

ml_table.to_parquet(
    ARTIFACT_PATH,
    index=False,
)

print(
    "Saved artifact:"
)

print(
    ARTIFACT_PATH
)

Saved artifact:
G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\01_joined\ml_table.parquet


In [22]:
saved_ml_table = pd.read_parquet(
    ARTIFACT_PATH
)

print(
    "Saved artifact shape:",
    saved_ml_table.shape
)

print(
    "Unique order_id:",
    saved_ml_table[
        "order_id"
    ].nunique()
)

print(
    "Duplicate order_id:",
    saved_ml_table[
        "order_id"
    ].duplicated().sum()
)

assert (
    saved_ml_table.shape
    == ml_table.shape
)

assert (
    saved_ml_table[
        "order_id"
    ].nunique()
    == len(saved_ml_table)
)

assert (
    saved_ml_table[
        "order_id"
    ].duplicated().sum()
    == 0
)

print(
    "\nSaved artifact validation passed."
)

Saved artifact shape: (99441, 44)
Unique order_id: 99441
Duplicate order_id: 0

Saved artifact validation passed.


In [23]:
print(
    "=" * 60
)

print(
    "NOTEBOOK 01 COMPLETE"
)

print(
    "=" * 60
)

print(
    f"Orders in ML table : {len(saved_ml_table):,}"
)

print(
    f"Columns            : {saved_ml_table.shape[1]}"
)

print(
    f"Unique order_id    : "
    f"{saved_ml_table['order_id'].nunique():,}"
)

print(
    f"Duplicate order_id : "
    f"{saved_ml_table['order_id'].duplicated().sum()}"
)

print(
    f"Artifact           : {ARTIFACT_PATH}"
)

NOTEBOOK 01 COMPLETE
Orders in ML table : 99,441
Columns            : 44
Unique order_id    : 99,441
Duplicate order_id : 0
Artifact           : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\01_joined\ml_table.parquet
